# One CSA scan through the pipeline

This notebook follows the included verified CSA/NASA example through image loading, structure detection, calibration, and warping. It uses the same reusable modules as the command-line pipeline.

In [ ]:
from pathlib import Path
import sys
import json
import matplotlib
_interactive = False
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
    _interactive = 'agg' not in str(matplotlib.get_backend()).lower()
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
def show_plot():
    if _interactive:
        plt.show()
    plt.close()

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'pyproject.toml').is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook from inside the FINAL ISIS repository')
sys.path[:0] = [str(ROOT), str(ROOT / 'src')]

film_path = ROOT / 'data/raw/csa_verified_ksh_1972322002235.png'
profile_path = ROOT / 'configs/film_calibration_profile.json'
assert film_path.is_file(), 'The verified CSA sample is missing from data/raw/'
assert profile_path.is_file()

from isis_research.image_io import load_image
from scripts.pipeline.extract_scan_structure import extract_structure, write_overlay
from scripts.pipeline.fit_frequency_axis import fit_from_profile, load_json
from scripts.pipeline.fit_height_axis import fit_from_profile as fit_height
from scripts.pipeline.standardize_film_only_512 import collapse_duplicate_fallback
from scripts.pipeline.warp_calibrated_scan import warp_one
from isis_research import ionogram

## 1. Load the raw film

The image is still in film coordinates. At this point the axes have no physical meaning.

In [ ]:
image = load_image(film_path)
plt.figure(figsize=(12, 7))
plt.imshow(image, cmap='gray', aspect='auto')
plt.title(f'Raw CSA scan: {film_path.name}')
plt.xlabel('film column')
plt.ylabel('film row')
show_plot()

## 2. Detect scan structure

This finds candidate film boundaries, vertical marker lines, and the horizontal ruling lattice. The detector does not assign frequencies or heights yet.

In [ ]:
structure = extract_structure(image)
structure_for_plot = dict(structure)
structure_for_plot['film_region'] = dict(structure['film_region'])
plot_path = ROOT / 'outputs/notebooks/01_structure.png'
plot_path.parent.mkdir(parents=True, exist_ok=True)
write_overlay(plot_path, image, structure_for_plot, film_path.name)
print(json.dumps({k: structure[k] for k in ('status', 'warnings', 'film_region', 'vertical_markers', 'horizontal_rulings')}, indent=2))
plt.figure(figsize=(12, 7))
plt.imshow(plt.imread(plot_path))
plt.axis('off')
show_plot()

## 3. Fit physical axes

The frequency fit matches detected marker positions to the stored film-format profile. The height fit uses the detected ruling lattice and the profile's learned scale and zero offset.

In [ ]:
profile = load_json(profile_path)
observed_markers = [item['x'] for item in structure['vertical_markers']['candidates']]
frequency = collapse_duplicate_fallback(fit_from_profile(observed_markers, image.shape, profile, metadata={}))
height = fit_height(structure, profile, frequency)
print('frequency:', frequency['status'], frequency.get('warnings', []))
print('height:', height['status'], height.get('warnings', []))

## 4. Warp to the digital ionogram grid

The result is normalized brightness in `(height, frequency)` orientation with a validity mask. Film darkness is still brightness here; consumers invert it once when they need signal-positive values.

In [ ]:
result, arrays = warp_one(image, frequency, height, structure, frequency_bins=512, height_bins=512)
if arrays is None:
    raise RuntimeError(f'Calibration did not produce a warp array: {result["status"]} — {result.get("reason", "see warnings")}')
print(json.dumps({k: result.get(k) for k in ('status', 'valid_coverage', 'frequency_min_mhz', 'frequency_max_mhz', 'height_min_km', 'height_max_km', 'warnings')}, indent=2))
plt.figure(figsize=(12, 7))
plt.imshow(
    arrays['warped'],
    cmap='gray',
    aspect='auto',
    vmin=0,
    vmax=1,
    origin='upper',
    extent=[float(arrays['frequency'][0]), float(arrays['frequency'][-1]), float(arrays['height'][-1]), float(arrays['height'][0])],
)
plt.title('Calibrated digital ionogram')
plt.xlabel('frequency (MHz)')
plt.ylabel('virtual height (km)')
show_plot()

In [ ]:
artifact_path = ROOT / 'outputs/notebooks/01_scan.npz'
ionogram.write(
    artifact_path, arrays['warped'], arrays['valid'], arrays['frequency'], arrays['height'],
    status=result['status'], route='film_only', confidence=float(result['confidence']),
    source={'raw_csa': film_path.name},
    provenance={'producer': 'notebooks/01_end_to_end_pipeline.ipynb'},
)
print(artifact_path)